## 1. EDA: первичный анализ маленького датасета

В этом разделе загружаем `student_data.csv`, смотрим структуру данных, пропуски, описательные статистики, корреляции и распределение целевой переменной `G3`.

### Импорт базовых библиотек

Подключаем `Path` для удобной работы с путями и `pandas` для чтения и анализа таблиц.

In [1]:
from pathlib import Path

import pandas as pd

### Загрузка данных

Читаем маленький CSV-файл из `data/raw` и выводим первые строки, чтобы проверить, что данные загрузились корректно.

In [2]:
DATA_PATH = Path("../data/raw/student_data.csv")
df = pd.read_csv(DATA_PATH)
df.head()

### Размер датасета

Смотрим количество строк и столбцов через `df.shape`.

In [3]:
df.shape

### Типы данных и непустые значения

`df.info()` показывает тип каждого столбца и помогает быстро увидеть, есть ли пропуски.

In [4]:
df.info()

### Проверка пропусков

Считаем пропущенные значения по каждому столбцу и сортируем столбцы по убыванию количества пропусков.

In [5]:
df.isna().sum().sort_values(ascending=False)

### Описательные статистики

`df.describe()` помогает понять диапазоны, средние значения, квартилли и возможные выбросы в числовых признаках.

In [6]:
df.describe()

### One-hot encoding для EDA

Преобразуем категориальные признаки в числовые `0/1`-столбцы, чтобы их можно было использовать в корреляциях и простых моделях.

In [7]:
df_encoding = pd.get_dummies(df, drop_first=True, dtype=int)
df_encoding.head()

### Импорт библиотек для графиков

Подключаем `seaborn` и `matplotlib` для визуализации данных.

In [8]:
import seaborn as sns
import matplotlib.pyplot as plt

### Корреляционная матрица Спирмена

Строим матрицу корреляций для числовых признаков. Метод Спирмена подходит для монотонных связей и не требует строго линейной зависимости.

In [ ]:
corr_matrix = df.corr(method="spearman", numeric_only=True)

plt.figure(figsize=(16, 14))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="Greens",
    cbar_kws={"label": "Сила корреляции (от -1 до 1)"},
)
plt.title("Корреляционная матрица Спирмена")
plt.show()

### Корреляции признаков с G3

Смотрим, какие числовые признаки сильнее всего связаны с итоговой оценкой `G3`.

In [10]:
corr_with_target = corr_matrix["G3"].drop("G3")
corr_with_target.sort_values(ascending=False).head(15).round(3)

### Положительные связи с G3

Отдельно выводим признаки, которые растут вместе с `G3`.

In [11]:
corr_with_target_plus = corr_with_target[corr_with_target > 0].sort_values(ascending=False).round(3)
corr_with_target_plus.head(6)

### Отрицательные связи с G3

Отдельно выводим признаки, которые связаны со снижением `G3`.

In [12]:
corr_with_target_minus = corr_with_target[corr_with_target < 0].sort_values(ascending=True).round(3)
corr_with_target_minus.head(7)

### Распределение целевой переменной

Строим гистограмму `G3`, чтобы увидеть, какие итоговые оценки встречаются чаще всего.

In [ ]:
df_g3 = df[["G3"]]

plt.figure(figsize=(8, 6))
sns.histplot(df_g3["G3"], bins=20, kde=True, color="skyblue")
plt.title("Распределение целевой переменной G3")
plt.xlabel("G3")
plt.ylabel("Количество студентов")
plt.show()

### Статистика по G3

Смотрим минимум, максимум, среднее, медиану и квартили целевой переменной.

In [14]:
df["G3"].describe()

### Средняя G3 по количеству прошлых неудач

Проверяем, как меняется средняя итоговая оценка при разных значениях `failures`.

In [15]:
df.groupby("failures")["G3"].mean()

### Средняя G3 по времени учебы

Сравниваем среднюю итоговую оценку для разных значений `studytime`.

In [16]:
df.groupby("studytime")["G3"].mean()

### Средняя G3 по планам на высшее образование

Проверяем различие средней `G3` между группами `higher = yes` и `higher = no`.

In [17]:
df.groupby("higher")["G3"].mean()

## 2. Обучение и сравнение на двух датасетах

В этом разделе сравниваем простую baseline-модель и `LinearRegression`. Проверяем два сценария: с промежуточными оценками `G1/G2` и без них.

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### Подготовка двух наборов признаков

Создаем два сценария: `X_with_grades` использует `G1` и `G2`, а `X_without_grades` исключает `G1`, `G2` и целевую переменную `G3`.

In [19]:
X_with_grades = df_encoding.drop(["G3"], axis=1)
X_without_grades = df_encoding.drop(["G1", "G2", "G3"], axis=1)
y = df_encoding["G3"]

### Train/test split для baseline с G1/G2

Делим данные на train и test для baseline-модели с промежуточными оценками.

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X_with_grades,
    y,
    test_size=0.3,
    random_state=42)

### Dummy baseline с G1/G2

Обучаем простую baseline-модель, которая предсказывает среднее значение `G3`.

In [21]:
dummy_model_with_grades = DummyRegressor(strategy="mean")
dummy_model_with_grades.fit(X_train, y_train)

y_pred_dummy_with_grades = dummy_model_with_grades.predict(X_test)
print(f"Dummy model with grades")

### Метрики Dummy baseline с G1/G2

Считаем `MAE`, `RMSE` и `R2` для baseline-модели.

In [ ]:
mae = mean_absolute_error(y_test, y_pred_dummy_with_grades)
rmse = mean_squared_error(y_test, y_pred_dummy_with_grades) ** 0.5
r2 = r2_score(y_test, y_pred_dummy_with_grades)

print("Dummy model with grades")
print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R2: {r2:.3f}")

### Train/test split для baseline без G1/G2

Делим данные на train и test для baseline-модели без промежуточных оценок.

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X_without_grades,
    y,
    test_size=0.3,
    random_state=42)

### Dummy baseline без G1/G2

Повторяем baseline для второго сценария. Для `DummyRegressor` результат ожидаемо почти такой же, потому что модель не использует признаки.

In [24]:
dummy_model_without_grades = DummyRegressor(strategy="mean")
dummy_model_without_grades.fit(X_train, y_train)

y_pred_dummy_without_grades = dummy_model_without_grades.predict(X_test)
print(f"Dummy model without grades")

### Метрики Dummy baseline без G1/G2

Считаем `MAE`, `RMSE` и `R2` для baseline-модели во втором сценарии.

In [ ]:
mae = mean_absolute_error(y_test, y_pred_dummy_without_grades)
rmse = mean_squared_error(y_test, y_pred_dummy_without_grades) ** 0.5
r2 = r2_score(y_test, y_pred_dummy_without_grades)

print("Dummy model without grades")
print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R2: {r2:.3f}")

### Cross-validation и LinearRegression

Подключаем cross-validation и простую линейную модель для сравнения с baseline.

In [26]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

### LinearRegression с G1/G2

Оцениваем линейную регрессию через 7-fold cross-validation по `MAE`.

In [27]:
model = LinearRegression()
linear_reg_mae_scores_with_grades = -cross_val_score(
    model,
    X_with_grades,
    y,
    cv=7,
    scoring='neg_mean_absolute_error')

linear_reg_mae_scores_with_grades.mean().round(3)

### LinearRegression без G1/G2

Повторяем cross-validation без промежуточных оценок, чтобы понять, насколько они важны для качества прогноза.

In [28]:
linear_reg_mae_scores_without_grades = -cross_val_score(
    model,
    X_without_grades,
    y,
    cv=7,
    scoring='neg_mean_absolute_error')

linear_reg_mae_scores_without_grades.mean().round(3)

### Визуальное сравнение MAE на основном датасете

Строим столбчатый график, чтобы сравнить baseline и `LinearRegression` на маленьком датасете.

In [ ]:
results = pd.DataFrame({
    "model": [
        "Dummy",
        "Dummy",
        "LinearRegression",
        "LinearRegression",
    ],
    "features": [
        "with G1/G2",
        "without G1/G2",
        "with G1/G2",
        "without G1/G2",
    ],
    "MAE": [
        3.691,
        3.691,
        linear_reg_mae_scores_with_grades.mean(),
        linear_reg_mae_scores_without_grades.mean(),
    ],
})

plt.figure(figsize=(10, 7))

sns.barplot(
    data=results,
    x="model",
    y="MAE",
    hue="features",
)

plt.title("Сравнение MAE: baseline и LinearRegression")
plt.xlabel("Модель")
plt.ylabel("MAE")
plt.ylim(0, results["MAE"].max() + 0.5)
plt.show()

### 2.2 Расширенный датасет `student_data_500k.csv`

Переходим к большому датасету. Полный EDA не повторяем, потому что структура похожа, но проверяем ключевые вещи для обучения.

In [30]:
DATA_PATH = Path("../data/raw/student_data_500k.csv")
df_500 = pd.read_csv(DATA_PATH)
df_500.head()

### One-hot encoding для 500k и удаление record_id

Кодируем категориальные признаки и удаляем `record_id`, потому что это технический идентификатор, а не полезный признак для модели.

In [31]:
df_500_encoding = pd.get_dummies(df_500, drop_first=True, dtype=int).drop(["record_id"], axis=1)

### Распределение G3 на 500k

Строим распределение целевой переменной на расширенном датасете.

In [ ]:
df_500_g3 = df_500[["G3"]]

plt.figure(figsize=(8, 6))
sns.histplot(df_500_g3["G3"], bins=20, kde=True, color="skyblue")
plt.title("Распределение целевой переменной G3 (500k)")
plt.xlabel("G3")
plt.ylabel("Количество студентов")
plt.show()

### Статистика G3 на 500k

Смотрим `count`, `mean`, `std`, `min`, `max` и квартили целевой переменной.

In [33]:
df_500_encoding["G3"].describe()

### Подготовка признаков для 500k

Создаем два сценария признаков: с `G1/G2` и без `G1/G2`. Целевая переменная сохраняется отдельно в `y_500`.

In [34]:
X_500_with_grades = df_500_encoding.drop(["G3"], axis=1)
X_500_without_grades = df_500_encoding.drop(["G1", "G2", "G3"], axis=1)
y_500 = df_500_encoding["G3"]

### Dummy baseline на 500k

Считаем baseline через cross-validation. Одного baseline достаточно, потому что `DummyRegressor` не использует признаки.

In [35]:
DummyRegressor_mae_scores_500_grades = -cross_val_score(
    DummyRegressor(),
    X_500_with_grades,
    y_500,
    cv=7,
    scoring='neg_mean_absolute_error')

DummyRegressor_mae_scores_500_grades.mean().round(3)

In [36]:
from sklearn.model_selection import cross_validate

### LinearRegression на 500k с G1/G2

Оцениваем линейную регрессию на расширенном датасете с промежуточными оценками.

In [37]:
model = LinearRegression()
scoring = {
    'mae': 'neg_mean_absolute_error',
    'mse': 'neg_mean_squared_error',
    'R2': 'r2'
}

linear_reg_scores_500_with_grades = cross_validate(
    model,
    X_500_with_grades,
    y_500,
    cv=7,
    scoring=scoring)

linear_metrics_500_with_grades = pd.DataFrame({
    'features': ['with G1/G2'],
    'MAE': [-linear_reg_scores_500_with_grades['test_mae'].mean()],
    'MSE': [-linear_reg_scores_500_with_grades['test_mse'].mean()],
    'R2': [linear_reg_scores_500_with_grades['test_R2'].mean()]
})

linear_metrics_500_with_grades.round(3)

### LinearRegression на 500k без G1/G2

Оцениваем линейную регрессию без промежуточных оценок, чтобы сравнить силу остальных признаков.

In [38]:
linear_reg_scores_500_without_grades = cross_validate(
    model,
    X_500_without_grades,
    y_500,
    cv=7,
    scoring=scoring)

linear_metrics_500_without_grades = pd.DataFrame({
    'features': ['without G1/G2'],
    'MAE': [-linear_reg_scores_500_without_grades['test_mae'].mean()],
    'MSE': [-linear_reg_scores_500_without_grades['test_mse'].mean()],
    'R2': [linear_reg_scores_500_without_grades['test_R2'].mean()]
})
linear_metrics_500_without_grades.round(3)

### Визуальное сравнение MAE на 500k

Строим график `MAE` для baseline и `LinearRegression` на расширенном датасете.

In [ ]:
results = pd.DataFrame({
    "model": [
        "Dummy",
        "Dummy",
        "LinearRegression",
        "LinearRegression",
    ],
    "features": [
        "with G1/G2",
        "without G1/G2",
        "with G1/G2",
        "without G1/G2",
    ],
    "MAE": [
        3.046,
        3.046,
        linear_metrics_500_with_grades["MAE"].mean(),
        linear_metrics_500_without_grades["MAE"].mean(),
    ],
})

plt.figure(figsize=(10, 7))

sns.barplot(
    data=results,
    x="model",
    y="MAE",
    hue="features",
)

plt.title("Сравнение MAE: baseline и LinearRegression (500k)")
plt.xlabel("Модель")
plt.ylabel("MAE")
plt.ylim(0, results["MAE"].max() + 0.5)
plt.show()

In [ ]:
results = pd.DataFrame({
    "model": [
        "LinearRegression",
        "LinearRegression",
    ],
    "features": [
        "with G1/G2",
        "without G1/G2",
    ],
    "R2": [
        linear_metrics_500_with_grades["R2"].mean(),
        linear_metrics_500_without_grades["R2"].mean(),
    ],
})

plt.figure(figsize=(10, 7))

sns.barplot(
    data=results,
    x="model",
    y="R2",
    hue="features",
)

plt.title("Сравнение R2: LinearRegression (500k)")
plt.xlabel("Модель")
plt.ylabel("R2")
plt.ylim(0, 1)
plt.show()

In [41]:
from sklearn.model_selection import train_test_split

train_index, test_index = train_test_split(
    df_500_encoding.index,
    test_size=0.2,
    random_state=42)

In [42]:
X_train_with = X_500_with_grades.loc[train_index]
X_test_with = X_500_with_grades.loc[test_index]

X_train_without = X_500_without_grades.loc[train_index]
X_test_without = X_500_without_grades.loc[test_index]

y_train = y_500.loc[train_index]
y_test = y_500.loc[test_index]

In [43]:
print(X_train_with.shape)
print(X_test_with.shape)
print(X_train_without.shape)
print(X_test_without.shape)
print(y_train.shape)
print(y_test.shape)

In [44]:
from sklearn.ensemble import HistGradientBoostingRegressor

boosting_model_with_grades = HistGradientBoostingRegressor(
    learning_rate=0.005,
    max_iter=50000,
    max_leaf_nodes=31,
    min_samples_leaf=50,
    l2_regularization=1.0,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=30,
    random_state=42,
)

In [45]:
boosting_model_with_grades.fit(X_train_with, y_train)
y_pred_with = boosting_model_with_grades.predict(X_test_with)
mae_with = mean_absolute_error(y_test, y_pred_with)
mse_with = mean_squared_error(y_test, y_pred_with)
rmse_with = mse_with ** 0.5
r2_with = r2_score(y_test, y_pred_with)

In [46]:
print(f"MAE:  {mae_with:.3f}")
print(f"MSE:  {mse_with:.3f}")
print(f"RMSE: {rmse_with:.3f}")
print(f"R2:   {r2_with:.3f}")

In [47]:
boosting_model_without_grades = HistGradientBoostingRegressor(
    learning_rate=0.005,
    max_iter=50000,
    max_leaf_nodes=31,
    min_samples_leaf=50,
    l2_regularization=1.0,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=30,
    random_state=42,
)

In [48]:
boosting_model_without_grades.fit(X_train_without, y_train)
y_pred_without = boosting_model_without_grades.predict(X_test_without)
mae_without = mean_absolute_error(y_test, y_pred_without)
mse_without = mean_squared_error(y_test, y_pred_without)
rmse_without = mse_without ** 0.5
r2_without = r2_score(y_test, y_pred_without)

In [49]:
print("Without G1/G2:")
print(f"MAE:  {mae_without:.3f}")
print(f"MSE:  {mse_without:.3f}")
print(f"RMSE: {rmse_without:.3f}")
print(f"R2:   {r2_without:.3f}")

In [50]:
print(boosting_model_with_grades.n_iter_)
print(boosting_model_without_grades.n_iter_)

## 3. Диагностика градиентного бустинга

В этом разделе обучаем `HistGradientBoostingRegressor`, смотрим метрики, кривые обучения, график факт/прогноз, остатки и permutation importance.

### Отдельная validation-выборка

Берем 10% от исходной train-части для наблюдения за обучением. Финальная test-часть остается отдельно для итоговой оценки.

In [51]:
train_inner_index, val_index = train_test_split(
    train_index,
    test_size=0.1,
    random_state=42,
)

X_train_curve_with = X_500_with_grades.loc[train_inner_index]
X_val_curve_with = X_500_with_grades.loc[val_index]

X_train_curve_without = X_500_without_grades.loc[train_inner_index]
X_val_curve_without = X_500_without_grades.loc[val_index]

y_train_curve = y_500.loc[train_inner_index]
y_val_curve = y_500.loc[val_index]

print(X_train_curve_with.shape, X_val_curve_with.shape)
print(y_train_curve.shape, y_val_curve.shape)

### Контрольные точки итераций

В начале метрики проверяются чаще, затем реже. Это позволяет увидеть динамику обучения без пересчета качества на каждой итерации.

In [56]:
iteration_checkpoints = (
    list(range(100, 1001, 100))
    + list(range(1500, 10001, 1000))
    + list(range(20000, 50001, 10000))
)

iteration_checkpoints

### Функция сбора MAE и R2

`warm_start=True` продолжает обучение существующей модели до следующей контрольной точки. Для каждой точки считаем `MAE` и `R2` на validation.

In [57]:
def collect_boosting_history(X_train, y_train, X_val, y_val, checkpoints):
    model = HistGradientBoostingRegressor(
        learning_rate=0.005,
        max_iter=checkpoints[0],
        max_leaf_nodes=31,
        min_samples_leaf=50,
        l2_regularization=1.0,
        early_stopping=False,
        warm_start=True,
        random_state=42,
    )

    history = []

    for iteration in checkpoints:
        model.set_params(max_iter=iteration)
        model.fit(X_train, y_train)

        y_val_pred = model.predict(X_val)
        val_mae = mean_absolute_error(y_val, y_val_pred)
        val_r2 = r2_score(y_val, y_val_pred)

        history.append({
            "iteration": iteration,
            "MAE": val_mae,
            "R2": val_r2,
        })

        print(
            f"iteration={iteration:>5} | "
            f"MAE={val_mae:.4f} | R2={val_r2:.4f}"
        )

    return model, pd.DataFrame(history)

### Кривая обучения с G1/G2

Этот блок может выполняться долго. Сначала смотрим сценарий с промежуточными оценками.

In [58]:
curve_model_with, history_with = collect_boosting_history(
    X_train_curve_with,
    y_train_curve,
    X_val_curve_with,
    y_val_curve,
    iteration_checkpoints,
)

history_with.tail()

iteration=  100 | MAE=2.0268 | R2=0.5523
iteration=  200 | MAE=1.4836 | R2=0.7600
iteration=  300 | MAE=1.2111 | R2=0.8405
iteration=  400 | MAE=1.0782 | R2=0.8736
iteration=  500 | MAE=1.0118 | R2=0.8885
iteration=  600 | MAE=0.9763 | R2=0.8962
iteration=  700 | MAE=0.9556 | R2=0.9006
iteration=  800 | MAE=0.9415 | R2=0.9036
iteration=  900 | MAE=0.9320 | R2=0.9057
iteration= 1000 | MAE=0.9241 | R2=0.9073
iteration= 1500 | MAE=0.9019 | R2=0.9119
iteration= 2500 | MAE=0.8800 | R2=0.9163
iteration= 3500 | MAE=0.8688 | R2=0.9186
iteration= 4500 | MAE=0.8613 | R2=0.9202
iteration= 5500 | MAE=0.8568 | R2=0.9211
iteration= 6500 | MAE=0.8540 | R2=0.9216
iteration= 7500 | MAE=0.8518 | R2=0.9221
iteration= 8500 | MAE=0.8501 | R2=0.9224
iteration= 9500 | MAE=0.8486 | R2=0.9227
iteration=20000 | MAE=0.8432 | R2=0.9238
iteration=30000 | MAE=0.8427 | R2=0.9240
iteration=40000 | MAE=0.8427 | R2=0.9240
iteration=50000 | MAE=0.8431 | R2=0.9239


,iteration,MAE,R2
18,9500,0.848626,0.922714
19,20000,0.843210,0.923828
20,30000,0.842700,0.923968
21,40000,0.842670,0.923970
22,50000,0.843080,0.923901


### Кривая обучения без G1/G2

Второй сценарий особенно важен: здесь проверяем, может ли нелинейная модель компенсировать отсутствие `G1` и `G2` за счет остальных признаков.

In [59]:
curve_model_without, history_without = collect_boosting_history(
    X_train_curve_without,
    y_train_curve,
    X_val_curve_without,
    y_val_curve,
    iteration_checkpoints,
)

history_without.tail()

iteration=  100 | MAE=2.5659 | R2=0.3043
iteration=  200 | MAE=2.2233 | R2=0.4798
iteration=  300 | MAE=1.9640 | R2=0.5926
iteration=  400 | MAE=1.7750 | R2=0.6645
iteration=  500 | MAE=1.6347 | R2=0.7118
iteration=  600 | MAE=1.5254 | R2=0.7469
iteration=  700 | MAE=1.4349 | R2=0.7736
iteration=  800 | MAE=1.3648 | R2=0.7934
iteration=  900 | MAE=1.3063 | R2=0.8097
iteration= 1000 | MAE=1.2600 | R2=0.8224
iteration= 1500 | MAE=1.1058 | R2=0.8626
iteration= 2500 | MAE=1.0035 | R2=0.8882
iteration= 3500 | MAE=0.9699 | R2=0.8965
iteration= 4500 | MAE=0.9527 | R2=0.9006
iteration= 5500 | MAE=0.9410 | R2=0.9035
iteration= 6500 | MAE=0.9338 | R2=0.9053
iteration= 7500 | MAE=0.9285 | R2=0.9065
iteration= 8500 | MAE=0.9249 | R2=0.9074
iteration= 9500 | MAE=0.9216 | R2=0.9082
iteration=20000 | MAE=0.9038 | R2=0.9124
iteration=30000 | MAE=0.8967 | R2=0.9139
iteration=40000 | MAE=0.8937 | R2=0.9147
iteration=50000 | MAE=0.8920 | R2=0.9151


,iteration,MAE,R2
18,9500,0.921556,0.908167
19,20000,0.903797,0.912353
20,30000,0.896737,0.913933
21,40000,0.893668,0.914663
22,50000,0.891953,0.915059


### Графики кривых MAE и R2

`MAE` должна уменьшаться, а `R2` увеличиваться. Плато показывает, что дополнительные итерации почти перестали улучшать validation-качество.

In [ ]:
history_with_plot = history_with.assign(features="with G1/G2")
history_without_plot = history_without.assign(features="without G1/G2")
history_all = pd.concat(
    [history_with_plot, history_without_plot],
    ignore_index=True,
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.lineplot(
    data=history_all,
    x="iteration",
    y="MAE",
    hue="features",
    marker="o",
    ax=axes[0],
)
axes[0].set_title("MAE на validation по итерациям")
axes[0].set_xlabel("Количество итераций")
axes[0].set_ylabel("MAE")

sns.lineplot(
    data=history_all,
    x="iteration",
    y="R2",
    hue="features",
    marker="o",
    ax=axes[1],
)
axes[1].set_title("R2 на validation по итерациям")
axes[1].set_xlabel("Количество итераций")
axes[1].set_ylabel("R2")
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

### Факт против прогноза и остатки

Левая диаграмма показывает близость прогнозов к идеальной диагонали. Правая показывает остатки: желательно, чтобы они располагались вокруг нуля без явного рисунка.

In [ ]:
residuals_with = y_test - y_pred_with

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sample_plot = pd.DataFrame({
    "actual": y_test,
    "predicted": y_pred_with,
}).sample(n=10000, random_state=42)

sns.scatterplot(
    data=sample_plot,
    x="actual",
    y="predicted",
    alpha=0.25,
    ax=axes[0],
)
axes[0].plot([0, 20], [0, 20], color="red", linestyle="--")
axes[0].set_title("Фактический G3 и прогноз с G1/G2")
axes[0].set_xlabel("Фактический G3")
axes[0].set_ylabel("Предсказанный G3")

residual_plot = pd.DataFrame({
    "predicted": y_pred_with,
    "residual": residuals_with,
}).sample(n=10000, random_state=42)

sns.scatterplot(
    data=residual_plot,
    x="predicted",
    y="residual",
    alpha=0.25,
    ax=axes[1],
)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title("Остатки модели с G1/G2")
axes[1].set_xlabel("Предсказанный G3")
axes[1].set_ylabel("Факт - прогноз")

plt.tight_layout()
plt.show()

### Permutation importance

Histogram boosting не предоставляет обычный `feature_importances_`. Поэтому используем permutation importance: перемешиваем один признак и измеряем, насколько ухудшилась `MAE`. Для контроля времени берем 10 000 test-строк и 3 повтора.

In [ ]:
from sklearn.inspection import permutation_importance

importance_sample = X_test_with.sample(n=10000, random_state=42)
y_importance_sample = y_test.loc[importance_sample.index]

permutation_result = permutation_importance(
    boosting_model_with_grades,
    importance_sample,
    y_importance_sample,
    scoring="neg_mean_absolute_error",
    n_repeats=3,
    random_state=42,
    n_jobs=-1,
)

feature_importance = pd.DataFrame({
    "feature": importance_sample.columns,
    "importance": permutation_result.importances_mean,
}).sort_values("importance", ascending=False)

feature_importance.head(15)

In [ ]:
plt.figure(figsize=(10, 7))

sns.barplot(
    data=feature_importance.head(15),
    x="importance",
    y="feature",
    color="steelblue",
)

plt.title("Топ-15 признаков по permutation importance")
plt.xlabel("Ухудшение MAE после перемешивания признака")
plt.ylabel("Признак")
plt.tight_layout()
plt.show()

## 4. Проверка надежности результата на 500k

Визуализация отдельного дерева сейчас не является обязательной частью проекта. Основная модель `HistGradientBoostingRegressor` хранит деревья во внутреннем формате, поэтому стандартная визуализация через `plot_tree` для нее не подходит.

На текущем этапе важнее проверить, насколько можно доверять высоким метрикам на датасете `500k`. Особенно важно проверить, нет ли дубликатов или одинаковых строк признаков, которые могли попасть одновременно в train и test.

Дальше проверяем:

1. Есть ли полные дубликаты строк в `student_data_500k.csv` без учета `record_id`.
2. Есть ли дубликаты по признакам, если убрать целевую переменную `G3`.
3. Есть ли пересечения одинаковых строк признаков между train и test.
4. Можно ли построить более честный split через `GroupShuffleSplit`.

In [67]:
df_500_model = df_500.drop(columns=["record_id"])

full_duplicates_count = df_500_model.duplicated().sum()
full_duplicates_share = df_500_model.duplicated().mean()

print(f"Количество полных дубликатов без record_id: {full_duplicates_count}")
print(f"Доля полных дубликатов: {full_duplicates_share:.4%}")

In [68]:
feature_columns_500 = df_500_model.drop(columns=["G3"]).columns

feature_duplicates_count = df_500_model.duplicated(subset=feature_columns_500).sum()
feature_duplicates_share = df_500_model.duplicated(subset=feature_columns_500).mean()

print(f"Количество дубликатов по признакам без G3: {feature_duplicates_count}")
print(f"Доля дубликатов по признакам без G3: {feature_duplicates_share:.4%}")

In [69]:
train_raw = df_500_model.loc[train_index]
test_raw = df_500_model.loc[test_index]

train_feature_hash = pd.util.hash_pandas_object(
    train_raw[feature_columns_500],
    index=False,
)

test_feature_hash = pd.util.hash_pandas_object(
    test_raw[feature_columns_500],
    index=False,
)

overlap_count = test_feature_hash.isin(set(train_feature_hash)).sum()
overlap_share = overlap_count / len(test_feature_hash)

print(f"Количество test-строк, признаки которых уже были в train: {overlap_count}")
print(f"Доля таких строк в test: {overlap_share:.4%}")

In [70]:
from sklearn.model_selection import GroupShuffleSplit

In [71]:
feature_groups = pd.util.hash_pandas_object(
    df_500_model[feature_columns_500],
    index=False,
)

group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42,
)

train_group_pos, test_group_pos = next(
    group_split.split(df_500_model, y_500, groups=feature_groups)
)

train_group_index = df_500_model.index[train_group_pos]
test_group_index = df_500_model.index[test_group_pos]

In [72]:
train_group_raw = df_500_model.loc[train_group_index]
test_group_raw = df_500_model.loc[test_group_index]

train_group_hash = pd.util.hash_pandas_object(
    train_group_raw[feature_columns_500],
    index=False,
)

test_group_hash = pd.util.hash_pandas_object(
    test_group_raw[feature_columns_500],
    index=False,
)

group_overlap_count = test_group_hash.isin(set(train_group_hash)).sum()
group_overlap_share = group_overlap_count / len(test_group_hash)

print(f"Количество пересечений train/test после GroupShuffleSplit: {group_overlap_count}")
print(f"Доля пересечений train/test после GroupShuffleSplit: {group_overlap_share:.4%}")
print(f"Размер train: {len(train_group_index)}")
print(f"Размер test: {len(test_group_index)}")

In [74]:
strict_feature_columns_500 = df_500_model.drop(columns=["G1", "G2", "G3"]).columns

strict_feature_groups = pd.util.hash_pandas_object(
    df_500_model[strict_feature_columns_500],
    index=False,
)

strict_group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42,
)

train_strict_pos, test_strict_pos = next(
    strict_group_split.split(df_500_model, y_500, groups=strict_feature_groups)
)

train_strict_index = df_500_model.index[train_strict_pos]
test_strict_index = df_500_model.index[test_strict_pos]

In [75]:
train_strict_raw = df_500_model.loc[train_strict_index]
test_strict_raw = df_500_model.loc[test_strict_index]

train_strict_hash = pd.util.hash_pandas_object(
    train_strict_raw[strict_feature_columns_500],
    index=False,
)

test_strict_hash = pd.util.hash_pandas_object(
    test_strict_raw[strict_feature_columns_500],
    index=False,
)

strict_overlap_count = test_strict_hash.isin(set(train_strict_hash)).sum()
strict_overlap_share = strict_overlap_count / len(test_strict_hash)

print(f"Количество строгих пересечений train/test: {strict_overlap_count}")
print(f"Доля строгих пересечений train/test: {strict_overlap_share:.4%}")
print(f"Размер strict train: {len(train_strict_index)}")
print(f"Размер strict test: {len(test_strict_index)}")

## 5. Повторная проверка LinearRegression на честных split

После проверки дубликатов стало ясно, что обычный random split завышал качество: одинаковые строки признаков попадали одновременно в train и test. Теперь переобучаем `LinearRegression` на более честных split:

- для сценария с `G1/G2` используем `train_group_index` и `test_group_index`;
- для сценария без `G1/G2` используем более строгие `train_strict_index` и `test_strict_index`.

Цель этого шага — быстро проверить, насколько изменятся метрики без пересечений между train и test.

In [ ]:
X_train_group_with = X_500_with_grades.loc[train_group_index]
X_test_group_with = X_500_with_grades.loc[test_group_index]
y_train_group_with = y_500.loc[train_group_index]
y_test_group_with = y_500.loc[test_group_index]

X_train_strict_without = X_500_without_grades.loc[train_strict_index]
X_test_strict_without = X_500_without_grades.loc[test_strict_index]
y_train_strict_without = y_500.loc[train_strict_index]
y_test_strict_without = y_500.loc[test_strict_index]

print(X_train_group_with.shape, X_test_group_with.shape)
print(X_train_strict_without.shape, X_test_strict_without.shape)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

linear_model_group_with = LinearRegression()
linear_model_group_with.fit(X_train_group_with, y_train_group_with)
y_pred_group_with = linear_model_group_with.predict(X_test_group_with)

linear_model_strict_without = LinearRegression()
linear_model_strict_without.fit(X_train_strict_without, y_train_strict_without)
y_pred_strict_without = linear_model_strict_without.predict(X_test_strict_without)

linear_fair_results = pd.DataFrame([
    {
        "dataset": "500k",
        "split": "GroupShuffleSplit",
        "model": "LinearRegression",
        "features": "with G1/G2",
        "MAE": mean_absolute_error(y_test_group_with, y_pred_group_with),
        "MSE": mean_squared_error(y_test_group_with, y_pred_group_with),
        "RMSE": mean_squared_error(y_test_group_with, y_pred_group_with) ** 0.5,
        "R2": r2_score(y_test_group_with, y_pred_group_with),
    },
    {
        "dataset": "500k",
        "split": "Strict GroupShuffleSplit",
        "model": "LinearRegression",
        "features": "without G1/G2",
        "MAE": mean_absolute_error(y_test_strict_without, y_pred_strict_without),
        "MSE": mean_squared_error(y_test_strict_without, y_pred_strict_without),
        "RMSE": mean_squared_error(y_test_strict_without, y_pred_strict_without) ** 0.5,
        "R2": r2_score(y_test_strict_without, y_pred_strict_without),
    },
]).round(3)

linear_fair_results

После запуска сравни эти метрики со старыми результатами `LinearRegression` на обычном random split. Если качество заметно упадет, значит прежний split действительно был слишком оптимистичным из-за повторяющихся строк.